In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

# ==========================================
# CONFIGURACIÓN
# ==========================================
storage_account = "adsldevcrm"
container_name = "raw"
scope = "databriksScope"

client_id = dbutils.secrets.get(scope=scope, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope, key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope=scope, key="sp-tenant-id")

silver_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/silver"
database_name = "crm_sales_db"

In [0]:
# ==========================================
# AUTENTICACIÓN
# ==========================================
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

spark.sql(f"USE {database_name}")

In [0]:
# ==========================================
# LEER BRONZE
# ==========================================
accounts_df = spark.table(f"{database_name}.bronze_accounts")
data_dictionary_df = spark.table(f"{database_name}.bronze_data_dictionary")
products_df = spark.table(f"{database_name}.bronze_products")
sales_pipeline_df = spark.table(f"{database_name}.bronze_sales_pipeline")
sales_teams_df = spark.table(f"{database_name}.bronze_sales_teams")

In [0]:
# ==========================================
# LIMPIAR ACCOUNTS
# ==========================================
accounts_df = accounts_df.dropDuplicates()
accounts_df = accounts_df.withColumn("account", F.trim(F.col("account")))
accounts_df = accounts_df.withColumn("sector", F.trim(F.col("sector")))
accounts_df = accounts_df.withColumn("office_location", F.trim(F.col("office_location")))
accounts_df = accounts_df.withColumn(
    "parent_company",
    F.when(F.col("parent_company").isNull(), F.lit("Independent")).otherwise(F.trim(F.col("parent_company")))
)

In [0]:
# ==========================================
# LIMPIAR DATA_DICTIONARY
# ==========================================
data_dictionary_df = data_dictionary_df.dropDuplicates()
data_dictionary_df = data_dictionary_df.withColumn("table", F.trim(F.col("table")))
data_dictionary_df = data_dictionary_df.withColumn("field", F.trim(F.col("field")))
data_dictionary_df = data_dictionary_df.withColumn("description", F.trim(F.col("description")))

# ==========================================
# LIMPIAR PRODUCTS
# ==========================================
products_df = products_df.dropDuplicates()
products_df = products_df.withColumn("product", F.trim(F.col("product")))
products_df = products_df.withColumn("series", F.trim(F.col("series")))

# ==========================================
# LIMPIAR SALES_TEAMS
# ==========================================
sales_teams_df = sales_teams_df.dropDuplicates()
sales_teams_df = sales_teams_df.withColumn("sales_agent", F.trim(F.col("sales_agent")))
sales_teams_df = sales_teams_df.withColumn("manager", F.trim(F.col("manager")))
sales_teams_df = sales_teams_df.withColumn("regional_office", F.trim(F.col("regional_office")))

# ==========================================
# LIMPIAR SALES_PIPELINE
# ==========================================
sales_pipeline_df = sales_pipeline_df.dropDuplicates()
sales_pipeline_df = sales_pipeline_df.withColumn("opportunity_id", F.trim(F.col("opportunity_id")))
sales_pipeline_df = sales_pipeline_df.withColumn("sales_agent", F.trim(F.col("sales_agent")))
sales_pipeline_df = sales_pipeline_df.withColumn("product", F.trim(F.col("product")))
sales_pipeline_df = sales_pipeline_df.withColumn(
    "account",
    F.when(F.col("account").isNull(), F.lit("unknown")).otherwise(F.trim(F.col("account")))
)
sales_pipeline_df = sales_pipeline_df.withColumn("deal_stage", F.trim(F.col("deal_stage")))

In [0]:
# ==========================================
# formatos fechas
# ==========================================
sales_pipeline_df = sales_pipeline_df.withColumn("engage_date", F.to_date(F.col("engage_date"), "yyyy-MM-dd"))
sales_pipeline_df = sales_pipeline_df.withColumn("close_date", F.to_date(F.col("close_date"), "yyyy-MM-dd"))

sales_pipeline_df = sales_pipeline_df.withColumn("close_value", F.col("close_value").cast("double"))


In [0]:
# ==========================================
# REVISIÓN
# ==========================================
display(accounts_df)
display(data_dictionary_df)
display(products_df)
display(sales_pipeline_df)
display(sales_teams_df)

In [0]:
# ==========================================
# GUARDAR TABLAS SILVER
# ==========================================
accounts_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.silver_accounts")
data_dictionary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.silver_data_dictionary")
products_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.silver_products")
sales_pipeline_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.silver_sales_pipeline")
sales_teams_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.silver_sales_teams")

# ==========================================
# GUARDAR RUTAS SILVER
# ==========================================
accounts_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/accounts")
data_dictionary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/data_dictionary")
products_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/products")
sales_pipeline_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/sales_pipeline")
sales_teams_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/sales_teams")

print("Silver terminado")